### Full pipeline

In [1]:
import pandas as pd,numpy as np,json
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe
from langchain.agents import create_agent
from pydantic import BaseModel,Field
from langchain.tools import tool
from langchain.agents.middleware import dynamic_prompt,ModelRequest
from langchain.agents.structured_output import ToolStrategy
from tqdm import tqdm
from time import time,sleep
import threading

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialize LLM

In [2]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=1)

In [3]:
@tool
def define_project_idea(pattern_description: str) -> str:
    """
    Tool: Define project idea based on Pattern Description
    """
    
    return f"""Define a realworld AI application idea to implement This AI pattern. Choose random relevant domain. It must be like actual application.
here is some domains: E-commerce, Healthcare, Finance, Education, Social Media, Travel, Real Estate, Entertainment, Food Delivery, Fitness, News Aggregation, Project Management, Customer Support, Event Planning, Job Recruitment, Online Learning, Personal Finance, Blogging Platform, Music Streaming, Video Sharing, Virtual Events, Remote Work Collaboration.
       Here is a pattern description: \n\n{pattern_description}"""

@tool
def define_project_architecture(project_idea: str) -> str:
    """
    Tool: Define project architecture based on project idea.
    """
    
    return f"Define the architecture for the AI application idea. Use suitable framworks and libraries to implement this AI App.\n Use frameworks libraries if need such as tensorflow, pytorch, jax, scikit-learn, lightgbm, xgboost, catboost, fastai, rapids-cuml, transformers, sentence-transformers, tokenizers, spacy, nltk, gensim, trl, accelerate, vllm, langchain, llama-index, chroma, faiss, weaviate, pinecone, milvus, qdrant, elasticsearch, haystack, pandas, numpy, dask, polars, datasets, pyarrow, langgraph, autogen, crewai, opendevin, dspy, semantic-kernel, langsmith, promptlayer, wandb, trulens, evals, guardrails-ai, pydantic, gradio, streamlit, openai, instructor-embedding, cohere, text2vec, clip, openclip, blip, blip2, lavis, diffusers, torchvision, opencv-python, fastapi, ray, bentoml, onnxruntime, tensorrt, tqdm, rich, loguru, python-dotenv, joblib, networkx, phidata, openaimultiswarm, lcel, memgpt, vectorhub, llmdatahub... Here is project idea:\n\n{project_idea}"

def generate_project_code(project_architecture: str) -> str:
    """
    Tool: Generate code based on project architecture.
    """
    
    return f"Generated code for project architecture, state the code file for realworld application,there can be multiple files,but i want all files in single code.Use suitable libraries and frameworks if needed.Don't add any comment or doc string.Just actual code only. Here is the project architecture:\n\n{project_architecture}"

# @tool
# def define_domain(description: str) -> str:
#     """
#     Tool: Define domain specific requirement prompt based on Pattern Description
#     """
    
#     return f"""Define a domain specific requirement prompt to implement This AI pattern. Choose random relevant domain. It must be like actual application.
# here is some domains: E-commerce, Healthcare, Finance, Education, Social Media, Travel, Real Estate, Entertainment, Food Delivery, Fitness, News Aggregation, Project Management, Customer Support, Event Planning, Job Recruitment, Online Learning, Personal Finance, Blogging Platform, Music Streaming, Video Sharing, Virtual Events, Remote Work Collaboration.
#        Here is a pattern description: \n\n{description}"""

# @tool
# def analyze_requirements(defined_domain: str) -> str:
#     """
#     Tool: Analyze project requirements / specs.
#     """
    
#     return f"Analysis result for requirements, use suitable framworks and libraries to implement this AI App.\n Use frameworks libraries if need such as tensorflow, pytorch, jax, scikit-learn, lightgbm, xgboost, catboost, fastai, rapids-cuml, transformers, sentence-transformers, tokenizers, spacy, nltk, gensim, trl, accelerate, vllm, langchain, llama-index, chroma, faiss, weaviate, pinecone, milvus, qdrant, elasticsearch, haystack, pandas, numpy, dask, polars, datasets, pyarrow, langgraph, autogen, crewai, opendevin, dspy, semantic-kernel, langsmith, promptlayer, wandb, trulens, evals, guardrails-ai, pydantic, gradio, streamlit, openai, instructor-embedding, cohere, text2vec, clip, openclip, blip, blip2, lavis, diffusers, torchvision, opencv-python, fastapi, ray, bentoml, onnxruntime, tensorrt, tqdm, rich, loguru, python-dotenv, joblib, networkx, phidata, openaimultiswarm, lcel, memgpt, vectorhub, llmdatahub... Here is desctiption with domain: \n\n{defined_domain}"

# @tool
# def generate_code(requirements: str) -> str:
#     """
#     Tool: Generate code based on project requirements / specs.
#     """
    
#     return f"Generated code for requirements, state the code file for realworld application,there can be multiple files,but i want all files in single code.Use suitable libraries and frameworks if needed. Here is the requirements:\n\n{requirements}"

@tool
def summarize_patterns(patterns: str) -> str:
    """
    Tool: Summarize ai design patterns list
    """
    
    return f"Generalize these AI patterns in a few sentences. Give the output as a pattern description. I want one generalized patterns. Suggest a name for it. AI Patterns:\n\n{patterns}"

class CodeGenOutput(BaseModel):
    """Result from code generation."""
    filename: str = Field(..., description="Proposed filename for the generated code")
    code: str = Field(..., description="The code body")
    explanation: str = Field(..., description="Short explanation of what the code does")

class SummaryGenOutput(BaseModel):
    """Result from code generation."""
    pattern_summary: str = Field(..., description="Concise summary of the AI Design pattern as a pattern description. Max 400 words.")


class Agent:
    def __init__(self, llm):
        self.llm = llm
        self.agent = None
    
    def create_code_generation_agent(self):
        self.agent = create_agent(
            self.llm,
            system_prompt="You are a AI Project code generation agent that generate code based on the provided AI Design pattern Description.",
            tools=[define_project_idea,define_project_architecture,generate_project_code],
            response_format=ToolStrategy(CodeGenOutput)
        )
    
    def create_patterns_summary_agent(self):
        self.agent = create_agent(
            self.llm,
            system_prompt="You are an AI pattern summarization agent that generate concise summaries for AI Design patterns.",
            tools=[summarize_patterns],
            response_format=ToolStrategy(SummaryGenOutput)
        )
    
    def invoke(self, msg: str) -> dict:
        inputs = {"messages": [{"role": "user", "content": msg}]}
        return self.agent.invoke(inputs,config={"recursion_limit": 100})

In [4]:
code_agent = Agent(llm)
code_agent.create_code_generation_agent()
summary_agent = Agent(llm)
summary_agent.create_patterns_summary_agent()

In [5]:
curated_clusters = json.load(open("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/cluster_results/curated_clusters_nov25.json","r"))
raw_patterns = pd.read_json("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run /extracted_patterns/all_patterns.json")
code_dir = "/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run/generated_code-v2"

In [6]:
def save_code_file(filename: str, code: str):
    filepath = os.path.join(code_dir, filename)
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w") as f:
        f.write(code)

In [7]:
import ast 

def check_code_file_syntax(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except SyntaxError as e:
        return False
    

In [14]:
### Generate pattern summaries
pattern_summaries = []

for curated_cluster in tqdm(curated_clusters, desc="Generating Pattern Summaries",ncols=80):
    patterns = raw_patterns[raw_patterns["Pattern Name"].isin(set(curated_cluster["l2_patterns"]))]
    msg = f"""{json.dumps(patterns.to_dict(orient="records"))}"""
    summary_response = summary_agent.invoke(msg)
    while "" == summary_response.get("structured_response",""):
        print("Retrying summary generation due to failure...")
        sleep(2)
        summary_response = summary_agent.invoke(msg)
    pattern_summary = summary_response["structured_response"].pattern_summary
    pattern_summaries.append({
        "cluster_id": curated_cluster["cluster_id"],
        "cluster_id": curated_cluster["short_name"],
        "pattern_summary": pattern_summary
    })


Generating Pattern Summaries:   0%|                      | 0/26 [00:00<?, ?it/s]

Retrying summary generation due to failure...


Generating Pattern Summaries: 100%|█████████████| 26/26 [08:21<00:00, 19.30s/it]


In [15]:
import json
json.dump(pattern_summaries, open("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run /extracted_patterns/pattern_summaries.json","w"), indent=4)

In [8]:
raw_patterns

,Pattern Name,Problem,Context,Solution,Result,Related Patterns,Category,Uses,Thinking
0,External Knowledge Augmentation,Large Language Models (LLMs) are bounded by th...,LLMs relying on fixed and parametric knowledge...,Augment LLMs with the capability to access ext...,LLMs can surpass traditional knowledge limitat...,"Domain-Specific Tool Integration, Robust Tool-...","Knowledge & Reasoning, Tools Integration","Accessing contemporary information, retrieving...",This pattern directly addresses a core limitat...
1,Domain-Specific Tool Integration,"LLMs, trained on general knowledge, often exhi...",LLMs needing to perform tasks requiring deep e...,Employ specific external tools like online cal...,"Mitigates the expertise gap in LLMs, enhancing...","External Knowledge Augmentation, Task Automati...","Tools Integration, Knowledge & Reasoning","Performing complex calculations, solving equat...",This pattern focuses on overcoming the LLM's i...
2,Task Automation via Tools,LLMs are fundamentally language processors and...,Users requiring LLMs to perform real-world act...,Integrate LLMs with external task automation t...,LLMs can facilitate the execution of external ...,"Task Decomposition and Planning, Parameter Ext...","Agentic AI, Tools Integration","Scheduling appointments, setting reminders, fi...",This pattern enables LLMs to act as agents in ...
3,Multimodal Interaction Augmentation,LLMs often struggle to consistently understand...,User interactions involving varied input types...,Deploy specialized tools like speech recogniti...,Improved understanding and response to a broad...,Tool-Augmented Response Synthesis,"AI–Human Interaction, Tools Integration","Understanding speech inputs, analyzing images,...",This pattern addresses the limitation of LLMs ...
4,Transparent Tool-Use Reasoning,The opaque 'black-box' nature of current LLMs ...,"LLM applications where interpretability, accou...",Utilize tool learning to enable LLMs to exhibi...,"More transparent LLM operations, allowing user...","Iterative Task Solving (with Feedback), Tool-A...","AI–Human Interaction, Agentic AI, LLM-specific","Explaining complex problem-solving steps, debu...",This pattern directly addresses a critical eth...
...,...,...,...,...,...,...,...,...,...
344,Denoising Sequence-to-Sequence Pretraining (BA...,Training a robust and versatile sequence-to-se...,Developing a general-purpose language model ca...,Pretrain an encoder-decoder transformer model ...,Produces a powerful pretrained seq2seq model (...,"Pretrained Component Integration, Encoder-Deco...",LLM-specific,"Generator component in RAG, general text gener...",This describes a specific and influential pret...
345,Fixed Document Index (during Fine-tuning),Updating the document encoder and rebuilding t...,Fine-tuning a retrieval-augmented generation (...,"Keep the document encoder (e.g., BERT_d) and t...",Reduces computational cost and training time s...,"End-to-End Joint Training, Dense Passage Retri...",MLOps,Optimizing the fine-tuning process for RAG mod...,This is a practical optimization strategy for ...
346,Thorough Decoding (RAGSequence),Accurately approximating arg max_y p(y|x) for ...,"Decoding for the RAGSequence model, which marg...",Run beam search independently for each of the ...,Provides a more accurate estimation of the mar...,"RAGSequence, Beam Search, Marginalization over...",LLM-specific,High-quality decoding for RAGSequence models w...,This is a specific algorithmic design for how ...
347,Fast Decoding (RAGSequence),The computational cost of Thorough Decoding fo...,"Decoding for the RAGSequence model, where a mo...","Make an approximation that p(y|x, z) = 0 if hy...",Significantly more efficient decoding for RAGS...,"RAGSequence, Thorough Decoding (RAGSequence), ...",LLM-specific,"Efficient decoding for RAGSequence models, par...","Similar to Thorough Decoding, this is an algor..."


In [19]:
raw_patterns["Pattern Name"].isin(set([
            "Robust Tool-Augmented Processing",
            "Human-in-the-Loop Data Collection Interface",
            "Reference-Supported Generation",
            "Trust Calibration through Transparency",
            "Prompt-based Defenses",
            "Knowledge Traceability and Correctability via Explicit Reasoning Paths",
            "Progressive Response Disclosure",
            "Task Automation via Tools",
            "Tool-Augmented Response Synthesis",
            "Transparent Tool-Use Reasoning",
            "Linear Scale (for Evaluation)",
            "Binary Score (for Evaluation)",
            "Separate LLM Extractor",
            "Verbalizer",
            "Model-Generated Guidelines (for evaluation)",
            "SelfCalibration",
            "Verbalized Score (for confidence calibration)",
            "Finetuning for Controlled Abstention",
            "Confidence Estimation (Self-rated Probabilities)",
            "Cultural Awareness (for cultural adaptation)",
            "Demonstration Ensembling (DENSE)",
            "Selecting Balanced Demonstrations (for bias mitigation)",
            "AttrPrompt",
            "Bias-Aware Design & Mitigation",
            "Debate-Style Evidence Aggregation"
            
        ])).sum()

np.int64(25)

In [9]:
patterns_descriptions = []
for curated_cluster in curated_clusters:
    patterns = raw_patterns[raw_patterns["Pattern Name"].isin(set(curated_cluster["l2_patterns"]))]
    file_count = 97
    msg = f"""{json.dumps(patterns.to_dict(orient="records"))}"""
    length = 0
    for root, dirs, files in os.walk(code_dir+f"/{curated_cluster['short_name']}"):
        length += len(files)
    
    file_count = max(file_count, len(curated_cluster["l2_patterns"]))
    if length >= file_count:
        print(f"Code files already generated for {curated_cluster['short_name']}, skipping...")
        continue

    print(f"Generating code for cluster ({curated_clusters.index(curated_cluster)+1}/{len(curated_clusters)}): ",curated_cluster["short_name"])
    
    t1 = time()
    t2 = time()
    for iter in range(file_count):
        summary = json.dumps(patterns.iloc[iter % len(patterns)].to_dict())
        if os.path.exists(f"{code_dir}/{curated_cluster['short_name']}/pattern_{iter+1}.py"):
            print(f" - code {iter+1} already exists, skipping.")
            continue
        print(f" - generating code {iter+1}",end="\r")
        retry_count = 0
        while True:
            code = code_agent.invoke(summary)
            retry_count += 1
            if code.get("structured_response") is not None:
                if check_code_file_syntax(code["structured_response"].code):
                    break
                else:
                    print(f" - code {iter+1} has syntax error, retrying...",end="\r")
            if retry_count >=5:
                print(" - failed to generate code after 5 retries, exiting.")
                exit()
        save_code_file(f"{curated_cluster['short_name']}/pattern_{iter+1}.py", code["structured_response"].code)
        print(f" - code {iter+1} generated in {time()-t2:.2f} seconds.")
        t2 = time()
    print(f" ✅ completed in {time()-t1:.2f} seconds.")

Code files already generated for Advanced LLM Prompting, skipping...
Code files already generated for Cross-lingual LLM Prompting, skipping...
Generating code for cluster (3/18):  LLM based Multimodal Generative Prompting
 - code 1 already exists, skipping.
 - code 2 already exists, skipping.
 - code 3 already exists, skipping.
 - code 4 already exists, skipping.
 - code 5 already exists, skipping.
 - code 6 already exists, skipping.
 - code 7 already exists, skipping.
 - code 8 already exists, skipping.
 - code 9 already exists, skipping.
 - code 10 already exists, skipping.
 - code 11 already exists, skipping.
 - code 12 already exists, skipping.
 - code 13 already exists, skipping.
 - code 14 already exists, skipping.
 - code 15 already exists, skipping.
 - code 16 already exists, skipping.
 - code 17 already exists, skipping.
 - code 18 already exists, skipping.
 - code 19 already exists, skipping.
 - code 20 already exists, skipping.
 - code 21 already exists, skipping.
 - code 22

<unknown>:19: SyntaxWarning: invalid escape sequence '\s'
<unknown>:59: SyntaxWarning: invalid escape sequence '\d'
<unknown>:93: SyntaxWarning: invalid escape sequence '\d'


 - code 78 generated in 95.28 seconds...
 - code 79 generated in 37.46 seconds.
 - code 80 generated in 26.21 seconds.
 - code 81 generated in 33.80 seconds.
 - code 82 generated in 69.04 seconds.
 - code 83 generated in 22.84 seconds.
 - code 84 generated in 33.40 seconds.
 - code 85 generated in 25.71 seconds.
 - code 86 generated in 41.59 seconds.
 - code 87 generated in 143.72 seconds..
 - code 88 generated in 49.56 seconds.
 - code 89 generated in 27.35 seconds.
 - code 90 generated in 29.19 seconds.
 - code 91 generated in 51.33 seconds.
 - code 92 generated in 39.02 seconds.
 - code 93 generated in 46.31 seconds.
 - code 94 generated in 37.47 seconds.
 - code 95 generated in 34.62 seconds.
 - code 96 generated in 30.52 seconds.
 - code 97 generated in 93.33 seconds.
 ✅ completed in 990.65 seconds.


In [ ]:
### Generate Code for summarized clusters(deprecated)

file_count = 30
patterns_descriptions = []
for curated_cluster in curated_clusters:
    patterns = raw_patterns[raw_patterns["Pattern Name"].isin(set(curated_cluster["l2_patterns"]))]
    
    msg = f"""{json.dumps(patterns.to_dict(orient="records"))}"""
    length = 0
    for root, dirs, files in os.walk(code_dir+f"/{curated_cluster['short_name']}"):
        length += len(files)
    
    if length >= file_count:
        print(f"Code files already generated for {curated_cluster['short_name']}, skipping...")
        continue

    print(f"Generating code for cluster ({curated_clusters.index(curated_cluster)+1}/{len(curated_clusters)}): ",curated_cluster["short_name"])
    retry_count = 0
    t1 = time()
    while True:
        summary = summary_agent.invoke(msg)
        if summary.get("structured_response") is not None:
            break
        retry_count += 1
        if retry_count >=5:
            print(" - failed to generate summary after 5 retries, exiting.")
            exit()
    t2 = time()
    for iter in range(file_count):
        if os.path.exists(f"{code_dir}/{curated_cluster['short_name']}/pattern_{iter+1}.py"):
            print(f" - code {iter+1} already exists, skipping.")
            continue
        print(f" - generating code {iter+1}",end="\r")
        retry_count = 0
        while True:
            code = code_agent.invoke(summary["structured_response"].pattern_summary)
            retry_count += 1
            if code.get("structured_response") is not None:
                if check_code_file_syntax(code["structured_response"].code):
                    break
                else:
                    print(f" - code {iter+1} has syntax error, retrying...",end="\r")
            if retry_count >=5:
                print(" - failed to generate code after 5 retries, exiting.")
                exit()
        save_code_file(f"{curated_cluster['short_name']}/pattern_{iter+1}.py", code["structured_response"].code)
        print(f" - code {iter+1} generated in {time()-t2:.2f} seconds.")
        t2 = time()
    print(f" ✅ completed in {time()-t1:.2f} seconds.")

Code files already generated for LLM Prompting, Evaluation, Alignment, skipping...
Code files already generated for Advanced MT Prompting, skipping...
Code files already generated for Cross-lingual LLM Prompting, skipping...
Code files already generated for Multimodal Generative Prompting, skipping...
Code files already generated for Integrating External Knlowladge, skipping...
Code files already generated for Integrating Knlowladge Graph, skipping...
Code files already generated for LLM Memory, Knowledge & Adaptation, skipping...
Code files already generated for Retrieval Augmented Generation(RAG) Optimization, handling hallucinations, skipping...
Code files already generated for Iterative Optimizations and ReAct, skipping...
Code files already generated for Tool Use, skipping...
Code files already generated for Planning, skipping...
Code files already generated for Enhanced User Intent Comprehension, skipping...
Code files already generated for Reasoning, Think Step by step, XoT, ski

In [10]:
summary["structured_response"]

NameError: name 'summary' is not defined

In [195]:
code['structured_response'].code

'"""Advanced Prompting System for a Customer Support Chatbot."""\n\nimport random\n\nclass SimulatedLLM:\n    """A simplified LLM simulator for demonstration purposes."""\n\n    def generate(self, prompt: str) -> str:\n        """Simulates LLM response generation based on the prompt content."""\n        print(f"\\n--- Simulated LLM Input ---\\n{prompt}\\n-----------------------------")\n        \n        response = ""\n        if "Role: Customer Support Agent" in prompt:\n            response += "Hello! I\'m here to help you as your dedicated customer support agent. "\n\n        if "Tone: empathetic, professional, concise" in prompt:\n            response += "I understand your concern and will do my best to provide a clear and helpful solution. "\n\n        if "Customer History: " in prompt or "Product Info: " in prompt:\n            response += "Looking at your records, "\n            if "Customer History: " in prompt:\n                history_start = prompt.find("Customer History: ")

In [197]:
for i in code["messages"]:
    print(i.name)
    print(i.content)

None
**Pattern Name:** Advanced Prompting and AI Behavior Shaping

**Pattern Description:** This meta-pattern encompasses various techniques and strategies for designing, optimizing, and orchestrating prompts to precisely control and enhance the behavior, performance, and ethical alignment of Large Language Models (LLMs) and other generative AI systems. It addresses challenges ranging from guiding basic task completion and ensuring output quality, to enabling complex reasoning and mitigating undesirable biases. Solutions involve structuring prompts with specific instructions, few-shot examples, dynamic contextual information, and even assigning roles or styles. Furthermore, it includes methods for automating prompt generation and optimization, using LLMs for self-evaluation or synthetic data filtering, and employing multi-step prompt chains to solve intricate problems. The aim is to maximize the AI's effectiveness, ensure factual consistency, reduce harmful outputs, and achieve robust,

In [11]:
def test_code_files_syntax(code_directory: str):
    error_count = 0
    correct_count = 0
    for root, dirs, files in os.walk(code_directory):
        for file in files:
            if file.endswith(".py"):
                filepath = os.path.join(root, file)
                with open(filepath, "r") as f:
                    code = f.read()
                    if not check_code_file_syntax(code):
                        print(f"Syntax error found in file: {filepath}")
                        os.remove(filepath)
                        error_count += 1
                    else:
                        correct_count += 1
    if error_count == 0:
        print("All code files have valid syntax.")
    else:
        print("="*50)
        print(f"Total files checked: {error_count + correct_count}")
        print(f"Total files with valid syntax: {correct_count}")
        print(f"Total files with syntax errors: {error_count}")

In [12]:
test_code_files_syntax(code_dir)

<unknown>:72: SyntaxWarning: invalid escape sequence '\}'
<unknown>:75: SyntaxWarning: invalid escape sequence '\}'
<unknown>:79: SyntaxWarning: invalid escape sequence '\}'
<unknown>:82: SyntaxWarning: invalid escape sequence '\}'
<unknown>:87: SyntaxWarning: invalid escape sequence '\}'
<unknown>:317: SyntaxWarning: invalid escape sequence '\S'


All code files have valid syntax.
